In [18]:
# Repositório da API: https://github.com/digitalinnovationone/santander-dev-week-2023-api
sdw2023_api_url = 'https://sdw-2023-prd.up.railway.app'

##EXTRAÇÃO:

In [19]:
import pandas as pd

df=pd.read_csv('SDW2023.csv')
user_ids=df['UserID'].tolist()
print(user_ids)

[4, 5, 6]


In [51]:
import requests
import json

def get_user(id):
  response=requests.get(f'{sdw2023_api_url}/users{id}')
  return response.json() if response.status_code==200 else None

users=[user for id in user_ids if (user := get_user(id)) is not None]
print(json.dumps(users,indent=2))

[]


##TRANSFORMAÇÃO:

In [36]:
!pip install openai

In [48]:
from google.colab import userdata

groq_api_key = userdata.get("groy_api_key")

In [49]:
from openai import OpenAI

client = OpenAI(
    api_key=groy_api_key,
    base_url="https://api.groq.com/openai/v1"
)

def generate_ia_news(user):
  completion = client.chat.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "O seu papel é ser um especialista em marketing bancário."},
        {"role": "user", "content": f"Crie uma mensagem para {user['name']} sobre a importância dos importância dos investimentos como renda financeira pessoal (máximo de 150 caracteres)"}
    ]
)
  return completion.choice[0].message.content.strip('\"')

for user in users:
  new=generate_ia_news(user)
  print(new)
  user['news'].append({"icon": "https://digitalinnovationone.github.io/santander-dev-week-2023-api/icons/credit.svg", "description": "news"})

## CARREGAMENTO:

In [50]:
def update_user(user):
  response = requests.put(f"{sdw2023_api_url}/users/{user['id']}", json=user)
  return True if response.status_code == 200 else False

for user in users:
  success = update_user(user)
  print(f"User {user['name']} updated? {success}!")